# Customer Churn Prediction and Customer Segmentation 

### Author
Rishabh Bhardwaj

## Project Overview

Customer churn is one of the biggest challenges faced by subscription-based businesses. This project predicts whether a customer is likely to leave the company using a Random Forest Classifier. Additionally, K-Means Clustering is applied to segment customers into meaningful business groups based on their tenure, spending behaviour, and churn probability.

The project demonstrates complete machine learning workflow including:

- Data Preprocessing
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Model Building
- Model Evaluation
- Hyperparameter Tuning
- Customer Segmentation
- Business Insights

In [ ]:

# Customer Churn Prediction
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

from sklearn.cluster import KMeans

# Plot settings
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

# Dataset

The dataset used in this project is the Telco Customer Churn Dataset.

It contains customer demographic information, service details, contract information, billing information, and churn status.

Target Variable:

- Churn Value
    - 1 → Customer Left
    - 0 → Customer Stayed

In [ ]:
df = pd.read_csv("../data/Telco_customer_churn.csv")

print("Dataset Loaded Successfully!\n")

print("Shape of Dataset:", df.shape)

df.head()

In [ ]:
# Basic Dataset Information

print("Dataset Information\n")
df.info()

print("\n\nMissing Values\n")
print(df.isnull().sum())

print("\n\nStatistical Summary\n")
display(df.describe())

print("\nCategorical Columns Summary\n")
display(df.describe(include='object'))

# Data Cleaning and Feature Selection

Remove unnecessary columns and prepare the dataset for machine learning.

In [ ]:
data = df.copy()

drop_columns = [
    'CustomerID',
    'Count',
    'Country',
    'State',
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
    'Churn Label',
    'Churn Score',
    'CLTV',
    'Churn Reason'
]

data.drop(columns=drop_columns, inplace=True)

data['Total Charges'] = pd.to_numeric(data['Total Charges'], errors='coerce')

# Fill missing values
data['Total Charges'].fillna(data['Total Charges'].median(), inplace=True)

print("Dataset Shape after Cleaning:", data.shape)

data.head()

# Categorical Feature Encoding

Convert categorical variables into numerical values using One-Hot Encoding.

In [ ]:
data = pd.get_dummies(data, drop_first=True)

print("Encoding completed successfully!")
print("New Shape:", data.shape)

data.head()

# Correlation Analysis

Visualize relationships between features using a correlation heatmap.

In [ ]:
plt.figure(figsize=(18, 14))

corr = data.corr(numeric_only=True)

sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")

plt.show()

# Random Forest Classification

# Feature and Target Selection

Separate independent features (X) and target variable (y).

In [ ]:
X = data.drop("Churn Value", axis=1)
Y = data["Churn Value"]

print("Features Shape :", X.shape)
print("Target Shape :", Y.shape)

# Train-Test Split

Split the dataset into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)

print(X_train.shape)
print(X_test.shape)

# Model Training

Train a Random Forest Classifier to predict customer churn.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, Y_train)

In [ ]:
Y_pred = rf.predict(X_test)

Y_pred[:10]

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(Y_test, Y_pred)

print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, Y_pred))

# Confusion Matrix

Visualize correct and incorrect predictions.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

# ROC Curve and AUC Score

Measure the classifier's discrimination capability.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_prob = rf.predict_proba(X_test)

churn_prob = y_prob[:,1]

fpr, tpr, threshold = roc_curve(Y_test, churn_prob)

auc_score = roc_auc_score(Y_test, churn_prob)

print("ROC AUC Score:", auc_score)

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc_score:.3f}")

plt.plot([0,1],[0,1],'r--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
# Cross Validation

from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    rf,
    X,
    Y,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:")
print(scores)

print("\nAverage Accuracy:")
print(scores.mean())

# Hyperparameter Tuning

Optimize the Random Forest model using GridSearchCV.

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    ),
    parameters,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, Y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid.best_score_)

In [ ]:
# Feature Importance
# Identify the most influential features affecting customer churn.
importance = pd.DataFrame({

    "Feature":X.columns,

    "Importance":rf.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(15)

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(

    x="Importance",

    y="Feature",

    data=importance.head(15),

    palette="viridis"

)

plt.title("Top 15 Important Features")

plt.tight_layout()

plt.savefig("../images/feature_importance.png")

plt.show()

# Customer Segmentation

Create a dataset for customer segmentation using churn probability and customer attributes.

In [ ]:
segmentation=pd.DataFrame({

"Tenure Months":df["Tenure Months"],

"Monthly Charges":df["Monthly Charges"],

"Total Charges":pd.to_numeric(df["Total Charges"],errors='coerce'),

"Churn Probability":rf.predict_proba(X)[:,1]

})

In [ ]:
segmentation=segmentation.dropna()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

scaled=scaler.fit_transform(segmentation)

# Elbow Method

Determine the optimal number of customer clusters.

In [ ]:
from sklearn.cluster import KMeans

wcss=[]

for i in range(1,11):

    kmeans=KMeans(

        n_clusters=i,

        random_state=42,

        n_init=10

    )

    kmeans.fit(scaled)

    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(7,5))

plt.plot(range(1,11),wcss,marker='o')

plt.xlabel("Number of Clusters")

plt.ylabel("WCSS")

plt.title("Elbow Method")

plt.savefig("../images/elbow_method.png")

plt.show()

## Elbow Method Interpretation

The Elbow Method was used to determine the optimal number of clusters for customer segmentation. The Within-Cluster Sum of Squares (WCSS) decreases significantly until **k = 3**, after which the reduction becomes more gradual.

Therefore, **3 clusters** were selected as the optimal number for K-Means clustering, balancing model simplicity and cluster quality.

# Final K-Means Clustering

Train the K-Means model using the selected number of clusters.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(scaled)

segmentation["Cluster"] = clusters

segmentation.head()

# Cluster Summary

Analyze the average characteristics of each customer segment.

In [ ]:
cluster_summary = segmentation.groupby("Cluster").mean()

cluster_summary

# Cluster Labeling

Assign meaningful business names to each customer segment.

In [ ]:
cluster_names = {
    0: "Budget Loyal Customers",
    1: "High Risk New Customers",
    2: "Loyal Premium Customers"
}

segmentation["Cluster Segment"] = segmentation["Cluster"].map(cluster_names)

segmentation.head()

In [ ]:
cluster_summary = segmentation.groupby("Cluster")[[
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Probability"
]].mean()

cluster_summary

In [ ]:
cluster_names = {
    0: "High Risk New Customers",
    1: "Budget Loyal Customers",
    2: "Premium Loyal Customers"
}

segmentation["Cluster Segment"] = segmentation["Cluster"].map(cluster_names)

# Customer Segmentation by Monthly Charges

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=segmentation,
    x="Monthly Charges",
    y="Churn Probability",
    hue="Cluster Segment",
    palette="Set2",
    s=40,
    alpha=0.65,
    edgecolor="white",
    linewidth=0.3
)

plt.title("Customer Segmentation by Monthly Charges", fontsize=16, fontweight="bold")
plt.xlabel("Monthly Charges", fontsize=12)
plt.ylabel("Churn Probability", fontsize=12)

plt.grid(alpha=0.3)

plt.legend(title="Customer Segment",
           bbox_to_anchor=(1.02,1),
           loc="upper left")

plt.tight_layout()

plt.savefig("../images/customer_segments_monthly.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

# Customer Segmentation by Tenure

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=segmentation,
    x="Tenure Months",
    y="Churn Probability",
    hue="Cluster Segment",
    palette="Set2",
    s=40,
    alpha=0.65,
    edgecolor="white",
    linewidth=0.3
)

plt.title("Customer Segmentation by Tenure", fontsize=16, fontweight="bold")
plt.xlabel("Tenure (Months)", fontsize=12)
plt.ylabel("Churn Probability", fontsize=12)

plt.grid(alpha=0.3)

plt.legend(
    title="Customer Segment",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "../images/customer_segments_tenure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Customer Segmentation by Total Charges

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=segmentation,
    x="Total Charges",
    y="Churn Probability",
    hue="Cluster Segment",
    palette="Set2",
    s=40,
    alpha=0.65,
    edgecolor="white",
    linewidth=0.3
)

plt.title("Customer Segmentation by Total Charges", fontsize=16, fontweight="bold")
plt.xlabel("Total Charges", fontsize=12)
plt.ylabel("Churn Probability", fontsize=12)

plt.grid(alpha=0.3)

plt.legend(
    title="Customer Segment",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "../images/customer_segments_total.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Conclusion

This project successfully predicts customer churn using a Random Forest Classifier and segments customers using K-Means Clustering.

## Results

- Accuracy: **77.79%**
- ROC-AUC Score: **0.8356**
- Cross Validation Accuracy: **77.92%**
- Best GridSearchCV Accuracy: **78.45%**

## Key Findings

- Total Charges is the most important feature influencing churn.
- Customers with longer tenure are less likely to churn.
- Monthly Charges significantly impact customer retention.
- Three distinct customer segments were identified:
  - Budget Loyal Customers
  - High Risk New Customers
  - Premium Loyal Customers

This project demonstrates an end-to-end machine learning workflow, including data preprocessing, visualization, model training, evaluation, hyperparameter tuning, and customer segmentation.